# 09 – ONNX Export & Dynamic Quantization Optimization

**Purpose:** Export our fine-tuned `DistilBERT` sequence classifier to ONNX format, apply dynamic INT8 quantization, and run comprehensive speed/footprint benchmarks.

This notebook demonstrates:
1. Exporting the PyTorch model to ONNX.
2. Generating dynamically quantized PyTorch and ONNX models.
3. Running inference speed tests comparing standard and optimized versions.
4. Reviewing model size, process RAM utilization, cold start load times, query latency, and throughput (QPS).

## 0. Setup and Environment

In [ ]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT / "SupportAI").exists():
    REPO_ROOT = REPO_ROOT / "SupportAI"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Run Exporter and Benchmarking Suite

We run the ModelOptimizer pipeline to output the ONNX and INT8 files and benchmark their performance.

In [1]:
from src.models.transformer.optimization import ModelOptimizer
from src.utils.constants import OUTPUT_DIR

model_dir = OUTPUT_DIR / "models" / "best_model"

optimizer = ModelOptimizer(model_dir, smoke_run=False)
optimizer.export_onnx()
optimizer.quantize_pytorch()
optimizer.quantize_onnx()
results = optimizer.benchmark_models()

print("Optimization and Benchmarking execution complete!")

ModuleNotFoundError: No module named 'src'

## 2. Model Optimization Benchmark Results Comparison

We load the generated metrics json file into a DataFrame to easily compare performance.

In [ ]:
benchmarks_path = OUTPUT_DIR / "metrics" / "optimization_benchmarks.json"
df = pd.read_json(benchmarks_path).T

print("Model optimization benchmark metrics comparison table:")
df

In [ ]:
# Export Phase Manifest
import json
from src.utils.artifacts import save_yaml
from src.api.app import get_git_commit

opt_metrics_path = REPO_ROOT / "outputs" / "metrics" / "optimization_benchmarks.json"
opt_metrics = {}
if opt_metrics_path.exists():
    with open(opt_metrics_path) as f:
        opt_metrics = json.load(f)

manifest = {
    "phase": "09_Optimization",
    "optimization_metrics": opt_metrics,
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_09_optimization.yaml")
print("YAML manifest saved successfully:")
print(manifest)
